# Self-Pruning Neural Network on CIFAR-10

The idea here is to build a network that figures out which of its own weights are useless — and learns to zero them out *while* training, not after.

Each linear layer gets learnable "gate" scores attached to its weights. During the forward pass, gates (passed through sigmoid) get multiplied onto the weights. If the optimizer decides a gate isn't worth keeping, it pushes it toward zero — effectively removing that connection.

We control how aggressively this happens using a sparsity penalty (λ) in the loss.

**Outline:**
1. `PrunableLinear` — custom linear layer with per-weight gate scores
2. Sparsity loss — L1 penalty on gate values
3. Training on CIFAR-10 with three λ values
4. Accuracy vs sparsity tradeoff analysis

In [ ]:
# install dependencies if not already present
import subprocess, sys

pkgs = ["torch", "torchvision", "matplotlib"]
subprocess.check_call([sys.executable, "-m", "pip", "install", *pkgs, "--quiet"])
print("all good")

In [ ]:
import math
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import torchvision
import torchvision.transforms as transforms

import matplotlib.pyplot as plt
import numpy as np

print(f"torch version: {torch.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")

## Part 1 — PrunableLinear Layer

This is a custom drop-in for `nn.Linear` that adds a learnable gate to every weight.
The key idea: instead of using the raw weight matrix, we multiply it element-wise by
sigmoid(gate_scores) before doing the linear op. Gates start at roughly 0.6 (not fully open,
not fully closed), and the optimizer can drive them toward 0 to prune connections it doesn't need.

Why sigmoid? It squashes gate values into (0, 1), so they can never go negative or explode.
A gate near 0 means that weight has almost no effect — it's effectively pruned.

In [ ]:
class PrunableLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features

        # weight matrix — same shape as nn.Linear
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias   = nn.Parameter(torch.zeros(out_features))

        # one gate score per weight — same shape
        # initialized at 0.4 so sigmoid(0.4) ≈ 0.6, gates start open but not saturated
        # (if we init too high like 5.0, sigmoid ≈ 1.0 and the sparsity gradient is nearly 0)
        self.gate_scores = nn.Parameter(torch.empty(out_features, in_features))

        self._init_params()

    def _init_params(self):
        # standard kaiming uniform, same as nn.Linear uses internally
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        nn.init.constant_(self.gate_scores, 0.4)

    def forward(self, x):
        gates = torch.sigmoid(self.gate_scores)   # values in (0, 1)
        pruned_w = self.weight * gates            # element-wise mask
        return F.linear(x, pruned_w, self.bias)   # x @ pruned_w.T + bias

    def get_gates(self):
        """Detached gate values for inspection, not for grad computation."""
        return torch.sigmoid(self.gate_scores).detach()


# quick sanity check — make sure shapes look right
test_layer = PrunableLinear(10, 5)
x_test = torch.randn(3, 10)
out_test = test_layer(x_test)
print(f"output shape: {out_test.shape}")  # should be (3, 5)
print(f"gate shape: {test_layer.get_gates().shape}")  # (5, 10)
print(f"initial gate mean: {test_layer.get_gates().mean():.4f}")  # should be ~0.6